# Language Proficiency & Social Integration Among International Students

-----

## Dataset Overview

This dataset explores the relationship between English language proficiency and social integration indicators for non-English speaking adults (aged 18–64) across U.S. counties in the year 2022.

It is built to support research into the question:

How does limited language proficiency impact social integration and confidence among international students?

-----

## Data Source

Source - U.S. Census Bureau — American Community Survey (ACS) 5-Year Estimates

Year - 2022

API - "https://api.census.gov/data/2022/acs/acs5"

Geography - 3,000 U.S. counties

License - Public Domain — U.S. Government Works


-----

## Tables Scraped

B16004 - Age × Language Spoken at Home × English Proficiency (18–64 age band)

B14001 - School Enrollment by Level

B19013 - Median Household Income

B05012 - Native vs. Foreign-Born Population

-----

## Column Reference

county - | string | County name |

state - | string | State name |

state_encoded - | int | Label-encoded state ID (for ML) |

total_non_english_speakers_18_64 - | int | Adults 18–64 speaking a non-English language at home |

foreign_born_population - | int | Total foreign-born residents in the county |

spanish_speakers_total - | int | Total Spanish speakers (18–64) |

spanish_eng_very_well - | int | Spanish speakers who speak English very well |

spanish_eng_well - | int | Spanish speakers who speak English well |

spanish_eng_not_well - | int | Spanish speakers who speak English not well |

spanish_eng_not_at_all - | int | Spanish speakers who speak no English |

indo_euro_speakers_total - | int | Total Indo-European language speakers (e.g. French, Hindi, Russian) |

indo_euro_eng_very_well - | int | Indo-European speakers — English very well |

indo_euro_eng_well - | int | Indo-European speakers — English well |

indo_euro_eng_not_well - | int | Indo-European speakers — English not well |

indo_euro_eng_not_at_all - | int | Indo-European speakers — no English |

asian_pac_speakers_total - | int | Total Asian/Pacific Island language speakers (e.g. Mandarin, Korean, Tagalog) |

asian_pac_eng_very_well - | int | Asian/Pacific speakers — English very well |

asian_pac_eng_well - | int | Asian/Pacific speakers — English well |

asian_pac_eng_not_well - | int | Asian/Pacific speakers — English not well |

asian_pac_eng_not_at_all - | int | Asian/Pacific speakers — no English |

other_lang_speakers_total - | int | Total speakers of other languages (Arabic, African languages, etc.) |

other_lang_eng_very_well - | int | Other language speakers — English very well |

other_lang_eng_well - | int | Other language speakers — English well |

other_lang_eng_not_well - | int | Other language speakers — English not well |

other_lang_eng_not_at_all - | int | Other language speakers — no English |

total_enrolled_in_school - | int | Total population enrolled in school (any level) |

enrolled_college_or_grad_school - | int | Population enrolled in college or graduate school |

median_household_income_usd - | int | Median household income in USD (social integration proxy) |

## Data Collection

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import warnings
import plotly.express as px
import plotly.figure_factory as ff
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 60)

## Data Scrapping

We used the ACS 5-Year estimates at county level to get ~3,000 rows of data.
Table "B16004" breaks down English proficiency (very well / well / not well / not at all) for the 18–64 age group across 4 language families directly measuring what affects international students.

In [2]:
BASE_URL = "https://api.census.gov/data"
YEAR = 2022
DATASET = "acs/acs5"

VARIABLES = ",".join(["NAME", "B16004_025E", "B16004_026E", "B16004_027E", "B16004_028E", "B16004_029E", "B16004_030E", "B16004_031E", "B16004_032E", "B16004_033E", "B16004_034E", "B16004_035E", "B16004_036E", "B16004_037E", "B16004_038E", "B16004_039E", "B16004_040E", "B16004_041E", "B16004_042E", "B16004_043E", "B16004_044E", "B16004_045E", "B14001_002E", "B14001_008E", "B19013_001E", "B05012_003E"])

print("Table: B16004 — Age × Language × English Proficiency")
print(f"Variables: {len(VARIABLES.split(','))} columns")

Table: B16004 — Age × Language × English Proficiency
Variables: 26 columns


In [3]:
"Fetches ACS data for all U.S. counties via the Census JSON API (requests) and allso uses BeautifulSoup to scrape the Census variable labels HTML page, providing a human-readable label lookup for every variable code. Later it returns df: Raw DataFrame with one row per county var_labels: Dictionary mapping variable codes to plain-English descriptions"

def scrape_census(year, dataset, variables):
    
    url = f"{BASE_URL}/{year}/{dataset}"
    params = {"get" : variables, "for" : "county:*", "in" : "state:*"}
    resp = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()
    raw = resp.json()
    df = pd.DataFrame(raw[1:], columns=raw[0])
    print(f"{df.shape[0]:,} rows × {df.shape[1]} columns fetched")

    label_url = f"{BASE_URL}/{year}/{dataset}/variables.html"

    var_labels = {}
    try:
        soup = BeautifulSoup(requests.get(label_url, timeout=20).text, "html.parser")
        table = soup.find("table")
        if table:
            for row in table.find_all("tr")[1:]:
                cols = row.find_all(["td", "th"])
                if len(cols) >= 2:
                    var_labels[cols[0].get_text(strip=True)] = cols[1].get_text(strip=True)
        print(f"{len(var_labels):,} variable labels scraped")
    except Exception as e:
        print(f"Label scrape failed: {e}")

    return df, var_labels

df_raw, var_labels = scrape_census(YEAR, DATASET, VARIABLES)

df_raw.head()

3,222 rows × 28 columns fetched
28,193 variable labels scraped


,NAME,B16004_025E,B16004_026E,B16004_027E,B16004_028E,B16004_029E,B16004_030E,B16004_031E,B16004_032E,B16004_033E,B16004_034E,B16004_035E,B16004_036E,B16004_037E,B16004_038E,B16004_039E,B16004_040E,B16004_041E,B16004_042E,B16004_043E,B16004_044E,B16004_045E,B14001_002E,B14001_008E,B19013_001E,B05012_003E,state,county
0,"Autauga County, Alabama",34424,646,514,97,35,0,273,240,33,0,0,315,182,99,16,18,161,7,37,117,0,13743,2430,68315,1552,01,001
1,"Baldwin County, Alabama",127387,4350,2039,1219,785,307,2096,1459,441,174,22,605,283,161,161,0,87,80,7,0,0,47838,7417,71039,8106,01,003
2,"Barbour County, Alabama",13587,958,597,106,226,29,97,69,0,28,0,97,47,20,30,0,61,30,31,0,0,5258,841,39712,776,01,005
3,"Bibb County, Alabama",13610,161,93,7,61,0,7,7,0,0,0,24,0,24,0,0,0,0,0,0,0,4488,461,50669,244,01,007
4,"Blount County, Alabama",31504,3053,1359,649,706,339,144,144,0,0,0,8,8,0,0,0,13,4,0,9,0,12727,1754,57440,2843,01,009


## Data Preprocessing

In [4]:
RENAME_MAP = {
    "NAME" : "county_state", #will be split below
    "B16004_025E" : "total_non_english_speakers_18_64",
    #Spanish
    "B16004_026E" : "spanish_speakers_total",
    "B16004_027E" : "spanish_eng_very_well",
    "B16004_028E" : "spanish_eng_well",
    "B16004_029E" : "spanish_eng_not_well",
    "B16004_030E" : "spanish_eng_not_at_all",
    #Indo-European (French, Hindi, Russian, Portuguese, etc.)
    "B16004_031E" : "indo_euro_speakers_total",
    "B16004_032E" : "indo_euro_eng_very_well",
    "B16004_033E" : "indo_euro_eng_well",
    "B16004_034E" : "indo_euro_eng_not_well",
    "B16004_035E" : "indo_euro_eng_not_at_all",
    #Asian / Pacific Island (Mandarin, Korean, Vietnamese, Tagalog, etc.)
    "B16004_036E" : "asian_pac_speakers_total",
    "B16004_037E" : "asian_pac_eng_very_well",
    "B16004_038E" : "asian_pac_eng_well",
    "B16004_039E" : "asian_pac_eng_not_well",
    "B16004_040E" : "asian_pac_eng_not_at_all",
    #Other languages (Arabic, Somali, Swahili, etc.)
    "B16004_041E" : "other_lang_speakers_total",
    "B16004_042E" : "other_lang_eng_very_well",
    "B16004_043E" : "other_lang_eng_well",
    "B16004_044E" : "other_lang_eng_not_well",
    "B16004_045E" : "other_lang_eng_not_at_all",
    #Enrollment
    "B14001_002E" : "total_enrolled_in_school",
    "B14001_008E" : "enrolled_college_or_grad_school",
    #Socioeconomic
    "B19013_001E" : "median_household_income_usd",
    "B05012_003E" : "foreign_born_population",
}

df = df_raw.rename(columns=RENAME_MAP)

#Splitted "County, State" into separate columns
df[["county", "state"]] = df["county_state"].str.split(", ", n=1, expand=True)
df.drop(columns=["county_state", "state", "county"], inplace=True)
df.insert(0, "county", df_raw["NAME"].str.split(", ").str[0])
df.insert(1, "state",  df_raw["NAME"].str.split(", ").str[1])

#Dropped FIPS code columns returned by the API
df.drop(columns=[c for c in df.columns if c in ["state", "county"] and df.columns.tolist().count(c) > 1],
        inplace=True, errors="ignore")
fips_cols = [c for c in df.columns if c not in RENAME_MAP.values() and c not in ["county", "state"]]
df.drop(columns=fips_cols, inplace=True, errors="ignore")

#Converted all numeric columns
num_cols = [c for c in df.columns if c not in ["county", "state"]]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

#Replaced Census sentinel value -666666666 with NaN
df.replace(-666666666, pd.NA, inplace=True)

#Label-encoded state for ML usability
le = LabelEncoder()
df.insert(2, "state_encoded", le.fit_transform(df["state"].fillna("Unknown")))

df.head()

,county,state,state_encoded,total_non_english_speakers_18_64,spanish_speakers_total,spanish_eng_very_well,spanish_eng_well,spanish_eng_not_well,spanish_eng_not_at_all,indo_euro_speakers_total,indo_euro_eng_very_well,indo_euro_eng_well,indo_euro_eng_not_well,indo_euro_eng_not_at_all,asian_pac_speakers_total,asian_pac_eng_very_well,asian_pac_eng_well,asian_pac_eng_not_well,asian_pac_eng_not_at_all,other_lang_speakers_total,other_lang_eng_very_well,other_lang_eng_well,other_lang_eng_not_well,other_lang_eng_not_at_all,total_enrolled_in_school,enrolled_college_or_grad_school,median_household_income_usd,foreign_born_population
0,Autauga County,Alabama,0,34424,646,514,97,35,0,273,240,33,0,0,315,182,99,16,18,161,7,37,117,0,13743,2430,68315,1552.0
1,Baldwin County,Alabama,0,127387,4350,2039,1219,785,307,2096,1459,441,174,22,605,283,161,161,0,87,80,7,0,0,47838,7417,71039,8106.0
2,Barbour County,Alabama,0,13587,958,597,106,226,29,97,69,0,28,0,97,47,20,30,0,61,30,31,0,0,5258,841,39712,776.0
3,Bibb County,Alabama,0,13610,161,93,7,61,0,7,7,0,0,0,24,0,24,0,0,0,0,0,0,0,4488,461,50669,244.0
4,Blount County,Alabama,0,31504,3053,1359,649,706,339,144,144,0,0,0,8,8,0,0,0,13,4,0,9,0,12727,1754,57440,2843.0


## EDA

In [5]:
print("DATASET SUMMARY")
print(f" Rows (counties) : {len(df):,}")
print(f" Columns : {len(df.columns)}")
print(f" Unique states : {df['state'].nunique()}")
print(f" Missing values : {df.isnull().sum().sum():,}")

DATASET SUMMARY
 Rows (counties) : 3,222
 Columns : 28
 Unique states : 52
 Missing values : 79


In [6]:
proficiency_totals = {"Very Well": df[["spanish_eng_very_well", "indo_euro_eng_very_well", "asian_pac_eng_very_well", "other_lang_eng_very_well"]].sum().sum(), "Well": df[["spanish_eng_well", "indo_euro_eng_well", "asian_pac_eng_well", "other_lang_eng_well"]].sum().sum(), "Not Well": df[["spanish_eng_not_well", "indo_euro_eng_not_well", "asian_pac_eng_not_well", "other_lang_eng_not_well"]].sum().sum(), "Not at All": df[["spanish_eng_not_at_all", "indo_euro_eng_not_at_all", "asian_pac_eng_not_at_all", "other_lang_eng_not_at_all"]].sum().sum(),}

prof_df = pd.DataFrame({"Proficiency": list(proficiency_totals.keys()), "Count": list(proficiency_totals.values())})

fig = px.bar(prof_df, x="Proficiency", y="Count", color="Proficiency", color_discrete_sequence=["#2ecc71", "#3498db", "#e67e22", "#e74c3c"], title="English Proficiency Distribution — All Non-English Speakers (18–64)")

fig.update_traces(text=[f"{v/1e6:.1f}M" for v in prof_df["Count"]], textposition="outside", showlegend=False)

fig.update_layout(xaxis_title="English Proficiency Level", yaxis_title="Total Population", template="simple_white")

fig.show()

In [7]:
lang_totals = {"Spanish": df["spanish_speakers_total"].sum(), "Indo-European": df["indo_euro_speakers_total"].sum(), "Asian/Pacific Island": df["asian_pac_speakers_total"].sum(), "Other Languages": df["other_lang_speakers_total"].sum()}

fig = px.pie(values=lang_totals.values(), names=lang_totals.keys(), color_discrete_sequence=["#e74c3c", "#3498db", "#f39c12", "#9b59b6"], title="Non-English Language Groups (18–64)")
fig.update_traces(textposition="inside", textinfo="label+percent", textfont=dict(size=11, color="white"))
fig.show()


In [8]:
state_fb = (df.groupby("state")["foreign_born_population"].sum().sort_values(ascending=False).head(15).reset_index())

fig = px.bar(state_fb, y="state", x="foreign_born_population", orientation="h", color="foreign_born_population", color_continuous_scale="Reds", title="Top 15 States by Foreign-Born Population", labels={"foreign_born_population": "Foreign-Born Population", "state": ""})

fig.update_xaxes(tickformat=",.0f")
fig.show()


## Exporting CSV

In [9]:
col_order = [
    #Identifiers
    "county", "state", "state_encoded",
    #Population
    "total_non_english_speakers_18_64",
    "foreign_born_population",
    #Spanish speakers
    "spanish_speakers_total",
    "spanish_eng_very_well", "spanish_eng_well",
    "spanish_eng_not_well", "spanish_eng_not_at_all",
    #Indo-European speakers
    "indo_euro_speakers_total",
    "indo_euro_eng_very_well", "indo_euro_eng_well",
    "indo_euro_eng_not_well", "indo_euro_eng_not_at_all",
    #Asian / Pacific Island speakers
    "asian_pac_speakers_total",
    "asian_pac_eng_very_well", "asian_pac_eng_well",
    "asian_pac_eng_not_well", "asian_pac_eng_not_at_all",
    #Other language speakers
    "other_lang_speakers_total",
    "other_lang_eng_very_well", "other_lang_eng_well",
    "other_lang_eng_not_well", "other_lang_eng_not_at_all",
    #Social integration indicators
    "total_enrolled_in_school",
    "enrolled_college_or_grad_school",
    "median_household_income_usd",
]

df_final = df[[c for c in col_order if c in df.columns]].copy()

#Dropped rows where all proficiency columns are null (unpopulated counties)
proficiency_cols = [c for c in df_final.columns if "_eng_" in c]
df_final.dropna(subset=proficiency_cols, how="all", inplace=True)
df_final.reset_index(drop=True, inplace=True)

OUTPUT = "lang_proficiency_social_integration_counties.csv"
df_final.to_csv(OUTPUT, index=False, encoding="utf-8-sig")

print(f" Exported: {OUTPUT}")
print(f" Rows: {len(df_final):,}")
print(f" Columns: {len(df_final.columns)}")
df_final.head()

 Exported: lang_proficiency_social_integration_counties.csv
 Rows: 3,222
 Columns: 28


,county,state,state_encoded,total_non_english_speakers_18_64,foreign_born_population,spanish_speakers_total,spanish_eng_very_well,spanish_eng_well,spanish_eng_not_well,spanish_eng_not_at_all,indo_euro_speakers_total,indo_euro_eng_very_well,indo_euro_eng_well,indo_euro_eng_not_well,indo_euro_eng_not_at_all,asian_pac_speakers_total,asian_pac_eng_very_well,asian_pac_eng_well,asian_pac_eng_not_well,asian_pac_eng_not_at_all,other_lang_speakers_total,other_lang_eng_very_well,other_lang_eng_well,other_lang_eng_not_well,other_lang_eng_not_at_all,total_enrolled_in_school,enrolled_college_or_grad_school,median_household_income_usd
0,Autauga County,Alabama,0,34424,1552.0,646,514,97,35,0,273,240,33,0,0,315,182,99,16,18,161,7,37,117,0,13743,2430,68315
1,Baldwin County,Alabama,0,127387,8106.0,4350,2039,1219,785,307,2096,1459,441,174,22,605,283,161,161,0,87,80,7,0,0,47838,7417,71039
2,Barbour County,Alabama,0,13587,776.0,958,597,106,226,29,97,69,0,28,0,97,47,20,30,0,61,30,31,0,0,5258,841,39712
3,Bibb County,Alabama,0,13610,244.0,161,93,7,61,0,7,7,0,0,0,24,0,24,0,0,0,0,0,0,0,4488,461,50669
4,Blount County,Alabama,0,31504,2843.0,3053,1359,649,706,339,144,144,0,0,0,8,8,0,0,0,13,4,0,9,0,12727,1754,57440
